# MTG Deck Card Predictor - Training Notebook

Train the Deck Card Predictor using a HeteroGNN + Cross-Attention Autoregressive Deck Constructor
with Gumbel-Softmax card selection and learned copy-count regression. Uses archetype-holdout
cross-validation with Jaccard similarity as the primary metric.

**Runtime**: Select `Runtime > Change runtime type > T4 GPU` before running.

## 1. Setup

In [ ]:
# Mount Google Drive (project lives here)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import sys

PROJECT_DIR = '/content/drive/MyDrive/mtg-graph'
os.chdir(PROJECT_DIR)

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(f'Working directory: {os.getcwd()}')

In [ ]:
# Install third-party dependencies only
!pip install -q torch-geometric sentence-transformers python-dotenv \
    beautifulsoup4 lxml spacy tqdm pyarrow

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Data Validation

In [ ]:
from src.config import GRAPH_PATH, METAGAME_PARQUET, DECKLISTS_PARQUET, DECK_NUM_ARCHETYPES
import pandas as pd

for name, path in [('Metagame', METAGAME_PARQUET),
                    ('Decklists', DECKLISTS_PARQUET), ('Graph', GRAPH_PATH)]:
    if path.exists():
        if path.suffix == '.parquet':
            rows = len(pd.read_parquet(path))
            print(f'  OK  {name}: {rows:,} rows')
        else:
            size_mb = path.stat().st_size / (1024 * 1024)
            print(f'  OK  {name}: {size_mb:.1f} MB')
    else:
        print(f'  MISSING  {name} — run: python -m src.ingest_all')

assert GRAPH_PATH.exists(), 'Graph not found! Run data ingestion first.'

# Check board column
dl = pd.read_parquet(DECKLISTS_PARQUET)
if 'board' in dl.columns:
    main_count = (dl['board'] == 'main').sum()
    side_count = (dl['board'] == 'side').sum()
    print(f'\n  Board split: {main_count:,} mainboard / {side_count:,} sideboard entries')
    print(f'  (Training uses mainboard only)')

# Show top N archetypes by meta share (matches training selection)
meta = pd.read_parquet(METAGAME_PARQUET)
latest = meta.sort_values('snapshot_date').groupby('archetype').last().reset_index()
latest = latest.dropna(subset=['meta_share_pct'])
ranked = latest.nlargest(DECK_NUM_ARCHETYPES, 'meta_share_pct')

print(f'\nTop {DECK_NUM_ARCHETYPES} archetypes by meta share (used for training):')
for i, (_, row) in enumerate(ranked.iterrows(), 1):
    print(f'  {i:2d}. {row["meta_share_pct"]:5.1f}%  {row["archetype"]}')
print(f'\nTotal archetypes available: {len(latest)}')

print('\nData ready for training.')

## 3. Hyperparameters

Tune these before training. Restart runtime and re-run from this cell to retrain with new values.

In [ ]:
# ── Hyperparameters (edit these to experiment) ──

# Model architecture
D_MODEL = 128            # GNN / cross-attention hidden dimension (default: 128)
D_MESSAGE = 128          # GNN message-passing hidden dimension (default: 128)
D_COUNT = 16             # Copy-count head bottleneck dim (default: 16)
NUM_ATTN_HEADS = 4       # Cross-attention heads (default: 4)
NUM_GNN_LAYERS = 2       # HeteroGNN layers (default: 2 = 2-hop message passing)
DROPOUT = 0.1            # Dropout rate (default: 0.1)

# Training
LEARNING_RATE = 0.0001   # Peak learning rate (default: 0.0001)
WEIGHT_DECAY = 0.00001   # L2 regularization (default: 0.00001)
NUM_EPOCHS = 50          # Max training epochs (early stopping may end sooner)
PATIENCE = 10            # Early stopping patience (default: 10)
RECENCY_DAYS = 30        # Use decklists from last N days (default: 30)

# Loss function
COUNT_LOSS_WEIGHT = 1.0  # Weight for count loss vs selection loss (default: 1.0)

# Gumbel-Softmax temperature annealing
GUMBEL_TAU_START = 1.0   # Initial temperature (default: 1.0)
GUMBEL_TAU_MIN = 0.1     # Minimum temperature (default: 0.1)
GUMBEL_DECAY = 0.01      # Exponential decay rate (default: 0.01)

# Cross-validation
N_VAL_ARCHETYPES = 3     # Archetypes held out per fold (default: 3)
N_CV_FOLDS = 5           # Number of CV folds (default: 5, set 1 for quick iteration)

# Learning rate schedule
WARMUP_EPOCHS = 5        # Linear warmup (default: 5, 0 to disable)
LR_SCHEDULER = 'cosine'  # Schedule after warmup: cosine | linear | none

print('Hyperparameters set.')
print(f'  Architecture: d_model={D_MODEL}, d_message={D_MESSAGE}, d_count={D_COUNT}, attn_heads={NUM_ATTN_HEADS}, layers={NUM_GNN_LAYERS}, dropout={DROPOUT}')
print(f'  Training:     lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY}, epochs={NUM_EPOCHS}, patience={PATIENCE}')
print(f'  Loss:         select + {COUNT_LOSS_WEIGHT} * count, Gumbel tau={GUMBEL_TAU_START}->{GUMBEL_TAU_MIN} (decay={GUMBEL_DECAY})')
print(f'  CV:           {N_CV_FOLDS} folds, {N_VAL_ARCHETYPES} held-out archetypes each')
print(f'  LR Schedule:  {WARMUP_EPOCHS}-epoch warmup -> {LR_SCHEDULER} decay')

## 4. Train Deck Predictor (GPU-Accelerated)

In [ ]:
import os, time
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
from src.train_deck import train_deck

start_time = time.time()
result = train_deck(
    device=device,
    d_model=D_MODEL,
    d_message=D_MESSAGE,
    d_count=D_COUNT,
    num_attn_heads=NUM_ATTN_HEADS,
    num_gnn_layers=NUM_GNN_LAYERS,
    dropout=DROPOUT,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    recency_days=RECENCY_DAYS,
    count_loss_weight=COUNT_LOSS_WEIGHT,
    gumbel_tau_start=GUMBEL_TAU_START,
    gumbel_tau_min=GUMBEL_TAU_MIN,
    gumbel_decay=GUMBEL_DECAY,
    n_val_archetypes=N_VAL_ARCHETYPES,
    n_cv_folds=N_CV_FOLDS,
    warmup_epochs=WARMUP_EPOCHS,
    lr_scheduler=LR_SCHEDULER,
)
elapsed = time.time() - start_time
print(f'\nTraining complete in {elapsed:.1f}s')

## 5. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fold_results = result['fold_results']
if fold_results:
    first_fold = fold_results[0]
    train_losses_raw = first_fold.get('train_losses', [])
    select_losses = [t.get('select_loss', 0) for t in train_losses_raw]
    count_losses = [t.get('count_loss', 0) for t in train_losses_raw]
    taus = [t.get('tau', 1.0) for t in train_losses_raw]
    val_metrics = first_fold.get('val_metrics_history', [])
    best_epoch = first_fold.get('best_epoch', 0)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Loss curves + temperature
    epochs = range(1, len(select_losses) + 1)
    ax1.plot(epochs, select_losses, alpha=0.7, label='Select Loss', color='#7F77DD')
    ax1.plot(epochs, count_losses, alpha=0.7, label='Count Loss', color='#1D9E75')
    ax1.axvline(x=best_epoch, color='r', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Training Loss')
    ax1.legend(loc='upper left')
    ax1_twin = ax1.twinx()
    ax1_twin.plot(epochs, taus, '--', color='#E8A838', alpha=0.6, label='Gumbel τ')
    ax1_twin.set_ylabel('Temperature')
    ax1_twin.legend(loc='upper right')
    ax1.grid(True, alpha=0.3)

    # Validation metrics
    val_epochs = [m.get('epoch', (i+1)*5) for i, m in enumerate(val_metrics)]
    val_prec = [m.get('precision', 0) for m in val_metrics]
    val_rec = [m.get('recall', 0) for m in val_metrics]
    val_jacc = [m.get('jaccard', 0) for m in val_metrics]
    val_exact = [m.get('exact_count_match', 0) for m in val_metrics]

    ax2.plot(val_epochs, val_jacc, 'o-', color='green', markersize=3, label='Jaccard')
    ax2.plot(val_epochs, val_prec, 's-', color='blue', markersize=3, label='Precision')
    ax2.plot(val_epochs, val_rec, '^-', color='orange', markersize=3, label='Recall')
    ax2.plot(val_epochs, val_exact, 'd-', color='purple', markersize=3, label='Exact Count')
    ax2.axvline(x=best_epoch, color='r', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Score'); ax2.set_title('Validation Metrics')
    ax2.set_ylim(0, 1); ax2.legend(); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    if len(fold_results) > 1:
        scores = [r['best_val_metric'] for r in fold_results]
        print(f'\nCross-Validation Summary:')
        print(f'  Jaccard: {sum(scores)/len(scores):.4f} +/- {(sum((x-sum(scores)/len(scores))**2 for x in scores)/len(scores))**0.5:.4f}')
        for r in fold_results:
            print(f'  Fold {r["fold"]}: Jaccard={r["best_val_metric"]:.4f} (val: {", ".join(r["val_archetypes"])})')

## 6. Embedding Diagnostics

Check whether archetype embeddings are differentiated after message passing.

In [ ]:
import torch.nn.functional as F

model = result['model']
data = result['data']
model.eval()

with torch.no_grad():
    x_dict = model.gnn(data)

arch_emb = x_dict['archetype']
card_emb = x_dict['card']
arch_names = data['archetype'].names

# Archetype embedding similarity
n_arch = arch_emb.shape[0]
cos_sim = F.cosine_similarity(arch_emb.unsqueeze(0), arch_emb.unsqueeze(1), dim=-1)

print(f'=== Archetype Embedding Cosine Similarity ({n_arch} archetypes) ===')
mask = ~torch.eye(n_arch, dtype=torch.bool, device=cos_sim.device)
off_diag = cos_sim[mask]
print(f'  Mean (excl self):  {off_diag.mean():.4f}')
print(f'  Min:               {off_diag.min():.4f}')
print(f'  Max:               {off_diag.max():.4f}')
print(f'  Std:               {off_diag.std():.4f}')

if off_diag.mean() > 0.95:
    print('  WARNING: Archetype embeddings are nearly identical!')
elif off_diag.mean() > 0.8:
    print('  Moderate similarity — some differentiation but could be better.')
else:
    print('  Good diversity — archetypes have distinct embeddings.')

print(f'\nPairwise similarities:')
header = f'{"":20s} ' + ' '.join(f'{n[:8]:>8s}' for n in arch_names)
print(header)
for i, name in enumerate(arch_names):
    row = f'{name:20s} ' + ' '.join(f'{cos_sim[i,j]:.4f}  ' for j in range(n_arch))
    print(row)

# Card embedding diversity
sample_n = min(100, card_emb.shape[0])
sample_cards = torch.randperm(card_emb.shape[0])[:sample_n]
sample_emb = card_emb[sample_cards]
card_cos = F.cosine_similarity(sample_emb.unsqueeze(0), sample_emb.unsqueeze(1), dim=-1)
card_mask = ~torch.eye(sample_n, dtype=torch.bool, device=card_cos.device)
card_off = card_cos[card_mask]
print(f'\n=== Card Embedding Diversity ({sample_n} random cards) ===')
print(f'  Mean pairwise similarity: {card_off.mean():.4f}')
print(f'  Std:                      {card_off.std():.4f}')

## 7. Evaluation & Dashboard

In [ ]:
from src.train_deck import evaluate_deck

evaluate_deck(result)

print(f"\nResults saved to: {result['run_dir']}")

In [ ]:
from src.visualize_deck import main as generate_deck_dashboard

generate_deck_dashboard(run_dir=result['run_dir'])

print(f"\nAll results in: {result['run_dir']}")
!ls -la "{result['run_dir']}"